# Bogotá Property Price Predictor
## Notebook 1 — Data Collection & Cleaning

> *Before any model can predict, the data must be trustworthy. This notebook builds that trust.*

**The question driving this project:** Can we estimate a residential property's price in Bogotá, Colombia from six basic characteristics — location, type, area, bedrooms, bathrooms, and parking?

This is the first of three notebooks. Here we build the foundation: 5,002 raw property listings scraped from Properati are transformed from messy HTML text into a clean, reliable dataset. Every cleaning decision is documented with its rationale so the pipeline is fully reproducible.

**This notebook covers:**
1. How the data was collected via web scraping
2. Initial data profiling — understanding what we have before touching it
3. A 10-step cleaning pipeline — parsing raw strings into usable features
4. Export of the cleaned dataset for EDA and modeling

| | |
|---|---|
| **Input** | `../data/properati.csv` — 5,002 raw listings scraped from properati.com.co |
| **Output** | `../data/cleaned_properati.csv` — clean rows ready for analysis |
| **Next** | Notebook 2 — Exploratory Data Analysis |


---
## 📦 Imports

In [1]:
import numpy as np
import pandas as pd

---
## 🕷️ Data Collection — Web Scraping

The dataset was scraped from [properati.com.co](https://www.properati.com.co/s/bogota-d-c-colombia/venta) using `requests` and `BeautifulSoup`. The scraper iterated over paginated results and extracted 7 fields per listing.

The scraping code is shown below as reference — the output CSV is already saved in `../data/properati.csv`.

```python
import requests
from bs4 import BeautifulSoup
from time import sleep
import pandas as pd

BASE_URL = "https://www.properati.com.co/s/bogota-d-c-colombia/venta"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}

PARAMS = {"propertyType": "studio,apartment,house,commercial,office"}

session = requests.Session()
session.headers.update(HEADERS)
data = []

for page in range(1, 10000):
    url = f"{BASE_URL}/{page}"
    try:
        response = session.get(url, params=PARAMS, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        print(f"Page error ({page}): {e}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    listings = soup.find_all("article")

    for item in listings:
        def get_text(tag, class_name):
            el = item.find(tag, class_=class_name)
            return el.get_text(strip=True) if el else None

        data.append({
            "price":     get_text("div",  "price"),
            "location":  get_text("div",  "location"),
            "type":      get_text("a",    "title"),
            "bedrooms":  get_text("span", "properties__bedrooms"),
            "bathrooms": get_text("span", "properties__bathrooms"),
            "area":      get_text("span", "properties__area"),
            "parking":   get_text("span", "properties__amenity__car_park"),
        })

    sleep(1)

pd.DataFrame(data).to_csv("../data/properati.csv", index=False)
```

---
## Initial Data Exploration

Before modifying a single value, we profile the raw dataset. Cleaning decisions must be grounded in what the data actually contains — not assumptions about what it should contain.

Two questions guide this section:
1. **What are we working with?** Shape, column types, and sample values.
2. **What is broken or incomplete?** Null counts, encoding issues, and structural problems.

The answers directly determine which cleaning steps are needed and in what order.


In [2]:
df = pd.read_csv("../data/properati.csv")
df.head()

,price,location,type,bedrooms,bathrooms,area,parking
0,Desde $ 859.500.000,"Usaquén, Zona Norte, Bogotá D.C, Cundinamarca",𝐃𝐔𝐀𝐋 𝟏𝟎𝟏 𝐇𝐎𝐔𝐒𝐄,2 - 4 habitaciones,3 - 4 baños,Desde 85 m²,NaN
1,Desde $ 466.475.500,"Suba, Zona Noroccidental, Bogotá D.C, Cundinam...",Hacienda Los Lagos Apartamentos,2 - 3 habitaciones,2 baños,Desde 54 m²,NaN
2,$ 13.047.900.000,"Niza, Suba, Zona Noroccidental, Bogotá D.C, Cu...",Oficina en Venta en Niza,NaN,NaN,2.538 m²,Parqueadero
3,$ 740.000.000,"Fontibón, Zona Occidental, Bogotá D.C, Cundina...",Local comercial en Venta en Fontibón,NaN,NaN,243 m²,NaN
4,$ 1.500.000.000,"Puente Aranda, Zona Centro, Bogotá D.C, Cundin...",Apartamento en Venta en Puente Aranda,3 habitaciones,"4,5 baños",205 m²,Parqueadero


In [3]:
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.info()

Shape: 5,002 rows × 7 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5002 entries, 0 to 5001
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   price      5002 non-null   object
 1   location   5002 non-null   object
 2   type       5002 non-null   object
 3   bedrooms   3540 non-null   object
 4   bathrooms  4342 non-null   object
 5   area       4462 non-null   object
 6   parking    2684 non-null   object
dtypes: object(7)
memory usage: 273.7+ KB


All columns are stored as `object` (string) — the scraper captured raw text directly from the HTML. Every field will need to be parsed.

In [4]:
df.isnull().sum().rename("null count").to_frame()

,null count
price,0
location,0
type,0
bedrooms,1462
bathrooms,660
area,540
parking,2318


**Missing value summary:**
| Column | Nulls | Notes |
|---|---|---|
| `parking` | 2,318 | Many listings simply don't advertise parking — treated as absence |
| `bedrooms` | 1,462 | Commercial/office listings often omit this |
| `bathrooms` | 660 | Less common to omit, filled with mode |
| `area` | 540 | Filled with column mean |

---
## The Cleaning Pipeline

The scraper returned raw HTML text for every field: prices like `"$ 1.200.000.000"`, areas like `"120 m²"`, property types like `"Apartamento en venta"`. No column is usable as-is.

Ten sequential steps convert this into a model-ready dataset. Steps are applied in a deliberate order — parsing must precede outlier removal, which must precede log-transformation. Each step includes the reasoning behind the decision, because reproducibility requires understanding *why*, not just *what*.


In [5]:
# --- Step 1: Drop promotional rows ---
# Rows 0–1 are sponsored ad listings injected at the top of every page,
# not real property entries. They contain misleading or aggregated data.
df = df.iloc[2:].reset_index(drop=True)

# --- Step 2: Parse price ---
# Raw format: "Desde $ 859.500.000" or "$ 740.000.000"
# Strip all non-numeric characters. "Desde $" (from-price) listings
# are accepted as minimum price — a reasonable approximation.
df["price"] = (
    df["price"]
    .str.replace(r"\D", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
    .astype("Int64")
)

# --- Step 3: Parse area ---
# Raw format: "Desde 85 m²" or "243 m²"
# Extract first numeric token. NaNs filled with the column mean (~148 m²)
# rather than dropping, to preserve the other fields of those listings.
df["area"] = (
    df["area"]
    .str.replace(r"\D", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
    .astype("Int64")
)
df["area"] = df["area"].fillna(df["area"].mean().round())

# --- Step 4: Parse location ---
# Raw format: "Usaquén, Zona Norte, Bogotá D.C, Cundinamarca"
# Keep only the neighborhood name (first token before the comma).
df["location"] = df["location"].str.split(",").str[0]

# --- Step 5: Parse property type ---
# Raw format: "Apartamento en Venta en Puente Aranda"
# The first word is always the property type.
df["type"] = df["type"].str.split().str[0]

# --- Step 6: Parse bedrooms ---
# Raw format: "2 - 4 habitaciones" or "3 habitaciones"
# For range values, take the lower bound (conservative estimate).
# NaNs filled with mode (most common bedroom count).
mode_bedrooms = df["bedrooms"].mode()[0]
df["bedrooms"] = df["bedrooms"].fillna(mode_bedrooms)
df["bedrooms"] = df["bedrooms"].str.split().str[0].astype(int)

# --- Step 7: Parse bathrooms ---
# Raw format: "4,5 baños" or "2 baños"
# Colombian decimal separator is a comma — replace with dot before casting.
# Supports half-bathrooms (e.g. 4.5). NaNs filled with mode.
mode_bathrooms = df["bathrooms"].mode()[0]
df["bathrooms"] = df["bathrooms"].fillna(mode_bathrooms)
df["bathrooms"] = (
    df["bathrooms"]
    .astype(str)
    .str.split().str[0]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# --- Step 8: Binary-encode parking ---
# A listing either advertises parking ("Parqueadero") or doesn't (NaN).
# Absence of the tag is treated as no parking — encoded as 0.
df["parking"] = df["parking"].notna().astype(int)

# --- Step 9: Remove outliers ---
# Cap the top 5% of price and area values. These outliers are likely luxury properties
df = df[df["area"]  < df["area"].quantile(0.95)]
df = df[df["price"] < df["price"].quantile(0.95)]

# --- Step 10: Log-transform price and area ---
# Both distributions are heavily right-skewed. Log transformation
# compresses the scale, making the target closer to normal (02_eda.ipynb) — a key
# assumption for regression models. Original columns are dropped.
df["log_area"]  = np.log(df["area"])
df["log_price"] = np.log(df["price"])
df = df.drop(columns=["price", "area"])

df.head()

,location,type,bedrooms,bathrooms,parking,log_area,log_price
1,Fontibón,Local,3,2.0,0,5.493061,20.422161
2,Puente Aranda,Apartamento,3,4.5,1,5.32301,21.128731
3,Puente Aranda,Casa,4,4.0,1,5.624018,21.639557
4,Niza,Casa,4,4.0,0,5.891644,21.716518
7,Chapinero Alto,Local,3,2.0,0,5.32301,20.903919


---
## What the Cleaning Produced

The pipeline retained the vast majority of listings while removing the records that would add noise rather than signal. The table below summarizes the final state of the dataset before it moves into analysis.


In [6]:
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Rows removed: {5002 - df.shape[0]:,} ({(5002 - df.shape[0]) / 5002 * 100:.1f}%)")
df.info()

Final shape: 4,509 rows × 7 columns
Rows removed: 493 (9.9%)
<class 'pandas.core.frame.DataFrame'>
Index: 4509 entries, 1 to 4998
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   location   4509 non-null   object 
 1   type       4509 non-null   object 
 2   bedrooms   4509 non-null   int64  
 3   bathrooms  4509 non-null   float64
 4   parking    4509 non-null   int64  
 5   log_area   4509 non-null   Float64
 6   log_price  4509 non-null   Float64
dtypes: Float64(2), float64(1), int64(2), object(2)
memory usage: 290.6+ KB


In [7]:
df.describe().round(3)

,bedrooms,bathrooms,parking,log_area,log_price
count,4509.000,4509.000,4509.000,4509.0,4509.0
mean,3.346,2.846,0.534,4.942,20.546
std,1.930,1.799,0.499,0.809,0.836
min,1.000,1.000,0.000,0.0,12.206
25%,3.000,2.000,0.000,4.369,20.0
50%,3.000,2.000,1.000,5.069,20.607
75%,3.000,4.000,1.000,5.624,21.162
max,30.000,56.500,1.000,6.593,22.056


---
## 💾 Export Cleaned Data

In [8]:
df.to_csv("../data/cleaned_properati.csv", index=False)
print("Saved → ../data/cleaned_properati.csv")

Saved → ../data/cleaned_properati.csv


---
## What Comes Next — Notebook 2

The cleaning pipeline has done its job: raw HTML text is now a structured, reliable dataset. But a clean dataset alone tells us nothing about the Bogotá real estate market.

**Notebook 2 (EDA)** investigates the market through the eyes of an analyst:
- How are prices distributed across the city? Is the distribution normal or skewed?
- Does location really drive price — and by how much?
- Which property characteristics correlate most strongly with price?

Every chart in Notebook 2 will produce a concrete modeling decision carried into Notebook 3.

> Continue to **[Notebook 2 — Exploratory Data Analysis](02_eda.ipynb)**
